In [1]:
import os
import re
import shutil
import random
from collections import defaultdict

import numpy as np
import h5py
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# ----------------------------- CONFIG -----------------------------
SRC_DIR  = "/kaggle/input/datasets/kangaonkaggle/synapse-abdomen-preprocessed/Synapse Abdomen Preprocessed"   # your existing Train/ Test/ Synapse Lists/
DEST_DIR = "Preprocessed"          # must be DIFFERENT from SRC_DIR

VAL_SIZE = 0.20          # fraction of TRAIN CASES that go to validation
SEED     = 42

TRANSFER_MODE    = "copy"   # "copy" | "move" | "symlink"  (Kaggle input is read-only -> use copy/symlink)
SAVE_VAL_VOLUMES = True     # also rebuild val 3D volumes (.npy.h5) for volume-level Dice
OVERWRITE        = False    # if DEST_DIR exists and is non-empty, refuse unless True
# ------------------------------------------------------------------

SLICE_RE = re.compile(r"^(case\d+)_slice(\d+)\.npz$")


def transfer(src, dst):
    if TRANSFER_MODE == "copy":
        shutil.copy2(src, dst)
    elif TRANSFER_MODE == "move":
        shutil.move(src, dst)
    elif TRANSFER_MODE == "symlink":
        if os.path.lexists(dst):
            os.remove(dst)
        os.symlink(os.path.abspath(src), dst)
    else:
        raise ValueError(f"Unknown TRANSFER_MODE: {TRANSFER_MODE}")


def group_slices_by_case(train_dir):
    """Return {case_id: [(slice_idx, filename), ...]} sorted by slice index."""
    groups = defaultdict(list)
    for fname in os.listdir(train_dir):
        m = SLICE_RE.match(fname)
        if m is None:
            print(f"[Warning] Unexpected file in Train dir, skipping: {fname}")
            continue
        groups[m.group(1)].append((int(m.group(2)), fname))
    for case in groups:
        groups[case].sort()
    return dict(groups)


def build_volume_from_slices(train_dir, slice_entries):
    """Stack 2D slices back into (H, W, D) volumes, matching how they were saved."""
    idxs = [i for i, _ in slice_entries]
    assert idxs == list(range(1, len(idxs) + 1)), \
        f"Slice indices are not contiguous 1..N (got {idxs[0]}..{idxs[-1]}, n={len(idxs)})"

    imgs, labs = [], []
    for _, fname in slice_entries:
        with np.load(os.path.join(train_dir, fname)) as d:
            imgs.append(d["image"])
            labs.append(d["label"])
    return np.stack(imgs, axis=2).astype(np.float32), np.stack(labs, axis=2).astype(np.uint8)


def write_list(path, names):
    with open(path, "w") as f:
        for n in names:
            f.write(n + "\n")


def main():
    random.seed(SEED)
    np.random.seed(SEED)

    src_train = os.path.join(SRC_DIR, "Train")
    src_test  = os.path.join(SRC_DIR, "Test")

    # ---- safety checks (no rmtree on purpose: never delete anything automatically) ----
    for p in (src_train, src_test):
        if not os.path.isdir(p):
            raise FileNotFoundError(f"Missing directory: {p}")
    if os.path.abspath(SRC_DIR) == os.path.abspath(DEST_DIR):
        raise ValueError("DEST_DIR must differ from SRC_DIR.")
    if os.path.isdir(DEST_DIR) and os.listdir(DEST_DIR) and not OVERWRITE:
        raise FileExistsError(f"{DEST_DIR} exists and is not empty. Set OVERWRITE=True or choose another path.")

    # ---- group slices by case ----
    groups = group_slices_by_case(src_train)
    if not groups:
        raise ValueError("No training slices found. Check the path / filename pattern.")
    case_ids = sorted(groups.keys())
    n_slices_total = sum(len(v) for v in groups.values())
    print(f"Found {len(case_ids)} training cases, {n_slices_total} slices total.")

    test_files = sorted(f for f in os.listdir(src_test) if f.endswith(".npy.h5"))
    test_cases = [f.replace(".npy.h5", "") for f in test_files]
    leaked = set(case_ids) & set(test_cases)
    assert not leaked, f"Cases appear in BOTH train and test: {sorted(leaked)}"

    # ---- split BY CASE ----
    train_cases, val_cases = train_test_split(case_ids, test_size=VAL_SIZE, random_state=SEED)
    train_cases, val_cases = sorted(train_cases), sorted(val_cases)
    assert not (set(train_cases) & set(val_cases)), "Train/val case overlap!"

    n_tr = sum(len(groups[c]) for c in train_cases)
    n_va = sum(len(groups[c]) for c in val_cases)
    print("-" * 40)
    print(f"Train: {len(train_cases)} cases, {n_tr} slices ({n_tr / n_slices_total:.1%} of slices)")
    print(f"Val:   {len(val_cases)} cases, {n_va} slices ({n_va / n_slices_total:.1%} of slices)")
    print(f"Test:  {len(test_cases)} cases (full volumes)")
    print(f"Val cases: {val_cases}")
    print("-" * 40)

    # ---- create output dirs ----
    dst_train = os.path.join(DEST_DIR, "Train")
    dst_val   = os.path.join(DEST_DIR, "Val")
    dst_valv  = os.path.join(DEST_DIR, "Val_vol")
    dst_test  = os.path.join(DEST_DIR, "Test")
    dst_lists = os.path.join(DEST_DIR, "Synapse Lists")
    dirs = [dst_train, dst_val, dst_test, dst_lists] + ([dst_valv] if SAVE_VAL_VOLUMES else [])
    for d in dirs:
        os.makedirs(d, exist_ok=True)

    # ---- val volumes first (must read from source BEFORE any 'move') ----
    if SAVE_VAL_VOLUMES:
        for c in tqdm(val_cases, desc="Building val volumes"):
            vol_img, vol_lab = build_volume_from_slices(src_train, groups[c])
            with h5py.File(os.path.join(dst_valv, f"{c}.npy.h5"), "w") as f:
                f.create_dataset("image", data=vol_img)
                f.create_dataset("label", data=vol_lab)

    # ---- transfer slices ----
    train_list, val_list = [], []
    for c in tqdm(train_cases, desc="Train slices"):
        for _, fname in groups[c]:
            transfer(os.path.join(src_train, fname), os.path.join(dst_train, fname))
            train_list.append(fname[:-4])
    for c in tqdm(val_cases, desc="Val slices"):
        for _, fname in groups[c]:
            transfer(os.path.join(src_train, fname), os.path.join(dst_val, fname))
            val_list.append(fname[:-4])

    # ---- transfer test volumes unchanged ----
    for f in tqdm(test_files, desc="Test volumes"):
        transfer(os.path.join(src_test, f), os.path.join(dst_test, f))

    # ---- list files ----
    write_list(os.path.join(dst_lists, "train.txt"), train_list)
    write_list(os.path.join(dst_lists, "val.txt"), val_list)
    write_list(os.path.join(dst_lists, "test_vol.txt"), test_cases)
    if SAVE_VAL_VOLUMES:
        write_list(os.path.join(dst_lists, "val_vol.txt"), val_cases)

    # ---- final sanity check on what was actually written ----
    n_tr_files = len([f for f in os.listdir(dst_train) if f.endswith(".npz")])
    n_va_files = len([f for f in os.listdir(dst_val) if f.endswith(".npz")])
    assert n_tr_files == n_tr and n_va_files == n_va, "File counts don't match expected slice counts!"
    print("\nDONE. Output at:", DEST_DIR)


if __name__ == "__main__":
    main()

Found 18 training cases, 2211 slices total.
----------------------------------------
Train: 14 cases, 1689 slices (76.4% of slices)
Val:   4 cases, 522 slices (23.6% of slices)
Test:  12 cases (full volumes)
Val cases: ['case0005', 'case0006', 'case0021', 'case0026']
----------------------------------------


Test volumes: 100%|██████████| 12/12 [00:26<00:00,  2.17s/it]


DONE. Output at: Preprocessed


In [2]:
main()

FileExistsError: Preprocessed exists and is not empty. Set OVERWRITE=True or choose another path.